### Create AnnData object for GSE17485 — Barker 2010 Lgr5⁺ stem cells across GI tract

- **Developed by:** Anna Maguza
- **Affiliation:** Faculty of Medicine, Würzburg University
- **Date of creation:** 7 May 2026
- **Last modified date:** 7 May 2026

Builds an AnnData (samples × genes) from 2 Agilent 2-channel microarray FE files at `LGR5_analysis_data/GSE17485/` (platform GPL4134). Barker 2010 design: comparative profiling of FACS-sorted Lgr5⁺ stem cells across small intestine, colon, and pyloric stomach.

⚠️ **Per-GSM tissue assignment is a best-guess** — only 2 GSMs are deposited here (`GSM436081`, `GSM436082`). Need to consult the GEO sample metadata (`https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE17485`) to know which GSM is which tissue. Both samples are FACS-sorted Lgr5+ per the publication design, so `lgr5_status='LGR5+'` for both. See `LGR5_data_folder_inventory.md` and `GSE_datasets_Lgr5_intestinal_stem_cells.md` (entry §5).

### Import packages

In [1]:
import os, sys
from datetime import datetime

import anndata as ad
import pandas as pd
import scanpy as sc

sys.path.insert(0, os.getcwd())
from utils.agilent_fe import load_agilent_study

/Users/am336941/uv_envs/pyenv313/.venv/lib/python3.11/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/Users/am336941/uv_envs/pyenv313/.venv/lib/python3.11/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/Users/am336941/uv_envs/pyenv313/.venv/lib/python3.11/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


### Per-sample metadata

In [2]:
DATA_DIR = '/Users/am336941/PhD/data/LGR5_analysis_data/GSE17485'

SAMPLE_PATHS = {
    'GSM436081': os.path.join(DATA_DIR, 'GSM436081.txt.gz'),
    'GSM436082': os.path.join(DATA_DIR, 'GSM436082.txt.gz'),
}

# Best-guess: GSM order = small intestine → pyloric stomach (Barker 2010 paper focuses on stomach Lgr5+ cells)
SAMPLE_META = {
    'GSM436081': dict(sample='Lgr5+_SI',      lgr5_status='LGR5+', lgr5_label_raw='Lgr5-EGFP+', condition='Lgr5-EGFP+', cell_type='Lgr5+ ISC',          tissue='small intestine',  GSE='GSE17485', organism='mus musculus', technology='Agilent 2-color microarray (GPL4134)', assay_modality='microarray'),
    'GSM436082': dict(sample='Lgr5+_pyloric', lgr5_status='LGR5+', lgr5_label_raw='Lgr5-EGFP+', condition='Lgr5-EGFP+', cell_type='Lgr5+ gastric stem', tissue='pyloric stomach',  GSE='GSE17485', organism='mus musculus', technology='Agilent 2-color microarray (GPL4134)', assay_modality='microarray'),
}

### Parse and build AnnData

In [3]:
adata = load_agilent_study(SAMPLE_PATHS, SAMPLE_META, value_column='LogRatio', aggregate='mean')
for col in ['sample', 'lgr5_status', 'lgr5_label_raw', 'condition', 'cell_type', 'tissue', 'GSE', 'organism', 'technology', 'assay_modality']:
    adata.obs[col] = adata.obs[col].astype('category')
adata

AnnData object with n_obs × n_vars = 2 × 27827
    obs: 'sample', 'lgr5_status', 'lgr5_label_raw', 'condition', 'cell_type', 'tissue', 'GSE', 'organism', 'technology', 'assay_modality'
    var: 'n_probes'
    uns: 'agilent_value_column', 'agilent_aggregate', 'agilent_n_channels'

### Save

In [4]:
timestamp = datetime.now().strftime('%d%m%Y_%H%M%S')
adata.uns['GSE'] = 'GSE17485'
adata.uns['publication'] = 'Barker N et al., Cell Stem Cell 6:25-36 (2010) — Lgr5+ve stem cells drive self-renewal in the stomach'
adata.uns['genome_reference'] = 'GPL4134 — Agilent Whole Mouse Genome Microarray 4x44K v2'
adata.uns['source_files'] = sorted([os.path.basename(p) for p in SAMPLE_PATHS.values()] + ['GPL4134_old_annotations.txt.gz'])
adata.uns['processing_history'] = {
    timestamp: 'AnnData created from 2 Agilent 2-channel FE files; LogRatio extracted; per-gene mean across probes; tissue best-guess by GSM order (verify against GEO metadata). | note: X holds log2(red/green) ratios per probe-aggregated gene. Both samples are Lgr5+ FACS-sorted; this study has no Lgr5- comparator deposited.',
}

out_dir = 'data/LGR5_analysis'
os.makedirs(out_dir, exist_ok=True)
out_path = f'{out_dir}/gut_mm_GSE17485_AM_{timestamp}_raw.h5ad'
adata.write_h5ad(out_path)
print(out_path, adata.shape)

data/LGR5_analysis/gut_mm_GSE17485_AM_07052026_233113_raw.h5ad (2, 27827)
